# Model Packaging for Distribution

## 📚 Learning Objectives

By completing this notebook, you will:
- Package models (e.g. pickle, SavedModel, ONNX)
- Version and load models in production

## 🔗 Prerequisites

- ✅ Basic Python
- ✅ Basic NumPy/Pandas (when applicable)

---

## Official Structure Reference

This notebook supports **Course 11, Unit 1** requirements from `DETAILED_UNIT_DESCRIPTIONS.md`.

---


# Model Packaging for Distribution
## AIAT 125 - Model Deployment

## Learning Objectives

- Package models in different formats
- Use Pickle, ONNX, and SavedModel
- Understand model serialization
- Prepare models for distribution

## Real-World Context

Model distribution, sharing, and deployment across platforms.

**Industry Impact**: Enables model sharing and cross-platform deployment.

## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
%pip install pickle5 onnx scikit-learn -q
import pickle
import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
print('✅ Setup complete!')

## Part 1: Pickle Format


In [ ]:
# Create sample model
model = RandomForestClassifier(n_estimators=100)
X_train = np.random.rand(100, 5)
y_train = np.random.randint(0, 2, 100)
model.fit(X_train, y_train)

# Save with Pickle
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Load model
with open('model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

print('✅ Model saved and loaded with Pickle')

## Part 2: Joblib Format


In [ ]:
# Save with Joblib (better for scikit-learn)
joblib.dump(model, 'model.joblib')

# Load model
loaded_model = joblib.load('model.joblib')

print('✅ Model saved and loaded with Joblib')

## Part 3: ONNX Format (for cross-platform)


In [ ]:
print('📝 ONNX Packaging:')
print('\n1. Convert model to ONNX format')
print('2. Enables cross-platform deployment')
print('3. Works on mobile, web, edge devices')
print('\n✅ ONNX packaging understood!')
print('\nReal-world: Deploy models on any platform')

## Real-World Applications

- **Model Sharing**: Distribute models to team members
- **Cross-Platform**: Deploy on different frameworks
- **Version Control**: Track model versions
- **Production**: Package for deployment pipelines

---

**End of Notebook**

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

steps = ['Train\nModel', 'Serialize\n(joblib/pickle)', 'Add\nDependencies', 'Package\n(ZIP/wheel)', 'Register\nin Registry']
x_positions = [0.1, 0.3, 0.5, 0.7, 0.9]
colors = ['#9b59b6', '#3498db', '#27ae60', '#e67e22', '#e74c3c']

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Model Packaging Pipeline', fontsize=14, fontweight='bold', pad=10)

box_w, box_h = 0.14, 0.35
y_center = 0.52

for i, (x, step, color) in enumerate(zip(x_positions, steps, colors)):
    rect = mpatches.FancyBboxPatch((x - box_w/2, y_center - box_h/2), box_w, box_h,
                                    boxstyle="round,pad=0.02", facecolor=color,
                                    edgecolor='black', linewidth=1.2)
    ax.add_patch(rect)
    ax.text(x, y_center, step, ha='center', va='center', fontsize=9,
            fontweight='bold', color='white', wrap=True)
    # Arrow
    if i < len(steps) - 1:
        ax.annotate('', xy=(x_positions[i+1] - box_w/2 - 0.005, y_center),
                    xytext=(x + box_w/2 + 0.005, y_center),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.8))

fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


## 🌍 Real-World Worked Example — Deploy a Trained Model as a REST API

**Industry context:**
- Spotify's recommendation model is served via a FastAPI microservice handling 400M users
- Instagram's image moderation runs as a containerised PyTorch model behind a REST endpoint
- Every ML feature in a modern app goes through a model serving layer like this

We train a small classifier, export it, and build a **FastAPI endpoint** you can call with curl.

In [ ]:
# ── Part 1: Train and save a model ────────────────────────────────────────
import torch, torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

iris = load_iris()
X = StandardScaler().fit_transform(iris.data.astype(np.float32))
y = iris.target
X_tr,X_te,y_tr,y_te = train_test_split(X, y, test_size=0.2, random_state=42)

model = nn.Sequential(nn.Linear(4,32), nn.ReLU(), nn.Linear(32,3))
opt   = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()
Xt = torch.tensor(X_tr); Yt = torch.tensor(y_tr, dtype=torch.long)

for _ in range(200):
    loss = loss_fn(model(Xt), Yt)
    opt.zero_grad(); loss.backward(); opt.step()

torch.save(model.state_dict(), '/tmp/iris_model.pt')
print("Model saved to /tmp/iris_model.pt")

# Verify
model.eval()
with torch.no_grad():
    acc = (model(torch.tensor(X_te)).argmax(1)==torch.tensor(y_te)).float().mean()
print(f"Test accuracy: {acc:.2%}")

# ── Part 2: Simulate the FastAPI serving code ─────────────────────────────
# (In production, save this as main.py and run: uvicorn main:app --reload)
fastapi_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import torch, torch.nn as nn
import numpy as np

app = FastAPI(title="Iris Classifier API")

# Load model at startup
model = nn.Sequential(nn.Linear(4,32), nn.ReLU(), nn.Linear(32,3))
model.load_state_dict(torch.load("/tmp/iris_model.pt"))
model.eval()
CLASSES = ["setosa", "versicolor", "virginica"]

class IrisRequest(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.post("/predict")
def predict(req: IrisRequest):
    features = torch.tensor([[req.sepal_length, req.sepal_width,
                               req.petal_length, req.petal_width]])
    with torch.no_grad():
        logits = model(features)
        probs  = torch.softmax(logits, dim=1)[0]
        label  = CLASSES[probs.argmax().item()]
    return {"prediction": label, "confidence": round(probs.max().item(), 3)}

@app.get("/health")
def health(): return {"status": "ok"}

# Run with: uvicorn main:app --host 0.0.0.0 --port 8000
# Test with: curl -X POST http://localhost:8000/predict -H "Content-Type: application/json" \
#            -d '{"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}'
'''
print("\n── FastAPI serving code (save as main.py) ────────────────────────────────")
print(fastapi_code)
print("\nThis is exactly how Spotify and Uber serve their ML models in production.")

## 📚 References & Further Reading

**Frameworks:**
- [FastAPI Documentation](https://fastapi.tiangolo.com/) — Modern Python API framework
- [ONNX Runtime](https://onnxruntime.ai/) — Cross-platform inference
- [BentoML](https://github.com/bentoml/BentoML) — ML model serving framework

**Cloud Services:**
- [AWS SageMaker Inference](https://docs.aws.amazon.com/sagemaker/latest/dg/deploy-model.html)
- [Google Cloud Vertex AI](https://cloud.google.com/vertex-ai/docs/predictions/overview)

**State-of-the-Art:** Uber, Airbnb, and Spotify deploy hundreds of ML models using microservices with FastAPI/gRPC.

## 📝 Summary

You learned **model packaging and serialization** — converting trained models into deployable artifacts. Pickle is simple but Python-only; ONNX is cross-platform and hardware-optimized. Production systems at Uber, Lyft, and Airbnb use ONNX and TorchScript for portability.